# Conditional expectation

_Probability & Statistics_

**The average of one variable once you know the other. Averages can be done in stages.**

The expectation gave the overall average of $X$.

     Conditional expectation $E[X \mid Y]$ is the average of $X$ once you know $Y$.

---

This notebook is a practice scaffold for the **Conditional expectation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPyWe simulate a world with three groups, give each group its own average for $Y$, then check two things: the per-group averages $E[Y \mid X=g]$, and the **law of total expectation** — that averaging those group means (weighted by how often each group appears) recovers the overall mean $E[Y]$. We build it in three steps.

### Step 1 — Simulate a grouped datasetEach example gets a group label `X` in {0, 1, 2}, drawn uniformly. The value `Y` is centred on a group-specific mean (1, 4, or 9) plus a little Gaussian noise. So `X` carries real information about `Y`: knowing the group shifts the expected value of `Y`.

In [ ]:
rng = np.random.default_rng(0)
n = 1_000_000

# Group label 0, 1, or 2, drawn uniformly.
X = rng.integers(0, 3, size=n)

# Y's mean depends on the group: group 0 -> 1, group 1 -> 4, group 2 -> 9.
means = np.array([1.0, 4.0, 9.0])
Y = means[X] + rng.normal(0, 1, size=n)

### Step 2 — Compute the conditional expectations$E[Y \mid X=g]$ is just the average of `Y` over the examples whose group is `g`. We select those examples with the boolean mask `X == g` and take their mean. The results should land close to the true group means (1, 4, 9), confirming that `Y[X == g].mean()` estimates the conditional expectation.

In [ ]:
# E[Y | X = g] for each group is the average of Y restricted to that group.
for g in range(3):
    group_mean = Y[X == g].mean()
    print("E[Y | X=%d] =" % g, round(group_mean, 3))

### Step 3 — Check the law of total expectationThe law of total expectation says $E[Y] = \sum_g P(X=g)\,E[Y \mid X=g]$: the overall mean is the weighted average of the group means, weighted by how likely each group is. We compute the group probabilities `pX`, the conditional means `condE`, combine them, and confirm the result matches `Y.mean()` directly.

In [ ]:
# P(X = g): the fraction of examples in each group.
pX = np.array([np.mean(X == g) for g in range(3)])

# E[Y | X = g]: the conditional mean for each group.
condE = np.array([Y[X == g].mean() for g in range(3)])

# Average the group means weighted by P(X=g) — this should equal E[Y].
total_expectation = np.sum(pX * condE)
print("E[E[Y|X]] =", round(total_expectation, 4))
print("E[Y]      =", round(Y.mean(), 4))

## Visualize the data & results_Does the law of total expectation recover E[Y] from group means?_

### Step 1 — Rebuild the grouped data for plottingWe regenerate the same simulated dataset (same seed, so it's identical) so this visualization cell stands on its own. `X` is the group label and `Y` is the group-dependent value, exactly as in the reference implementation above.

In [ ]:
# Group label X in {0,1,2}; Y mean depends on the group (1, 4, 9).
rng = np.random.default_rng(0)
n = 1_000_000
X = rng.integers(0, 3, size=n)
means = np.array([1.0, 4.0, 9.0])
Y = means[X] + rng.normal(0, 1, size=n)

### Step 2 — Plot the group means against the overall meanWe compute each group's conditional mean and the overall mean, then draw them as bars. The first three bars are $E[Y \mid X=g]$; the last is the overall $E[Y]$, which sits between them — a visual reminder that the overall average is a blend of the group averages.

In [ ]:
# Conditional mean per group, plus the overall mean.
condE = [Y[X == g].mean() for g in range(3)]
overall = Y.mean()
vals = condE + [overall]

labels = ['E[Y|X=0]', 'E[Y|X=1]', 'E[Y|X=2]', 'overall E[Y]']
colors = ['#4ea1ff', '#4ea1ff', '#4ea1ff', '#ffb454']

plt.bar(labels, vals, color=colors)
plt.title('E[Y | X=g] per group vs overall E[Y]')
plt.ylabel('expected value')
plt.show()